In [56]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
from utils_tool import utils,myplot

from grid_world.data_parser import DataParser
from grid_world.grid_world import GridWorld

import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv(r'wifi_track_data\dacang\track_data\dacang_track_data_final_{date}.csv')
df.t = pd.to_datetime(df.t)
mac_list = df.m.unique()

#get path & pos data
df_wifipos = pd.read_csv('wifi_track_data/dacang/pos_data/wifi_pos_new_0925.csv')
df_path = pd.read_csv('wifi_track_data/dacang/pos_data/path_pos.csv')

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Initialize grid world

In [57]:
env_folder_path = r'wifi_track_data/dacang/grid_data/env_imgs/40_30'
#feature_folder_path = r"wifi_track_data\dacang\grid_data\features_grid\0130_40x30"
expert_traj_path = r"wifi_track_data\dacang\track_data\trajs_sliced_0925_40x30.csv"

world = GridWorld(environments_img_folderPath=env_folder_path,
                  expert_traj_filePath= expert_traj_path,
                  width=40,height=30)

Get experts trajs num:7353
trajs all avg length: 23


parsing environments from folder:: 100%|██████████| 30/30 [00:06<00:00,  4.75it/s]


In [58]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from DMEIRL.value_iteration import value_iteration
import numpy as np
import os
from utils_tool import utils

### Show Dynamics
# Figures 14 (b): Transfer probabilities in upward moving behavior

In [59]:
world.dynamics.shape

(1200, 5, 1200)

In [60]:
world.ShowDynamics(0)

In [61]:
world.ShowDynamics(1)

In [62]:
world.ShowDynamics(2)

In [63]:
world.ShowDynamics(3)

In [64]:
world.ShowDynamics(4)

### Figures 13 (a)(b): Environments and Features

In [65]:
world.ShowEnvironments()

In [66]:
world.features_arr[1]

array([0.11841332, 0.0195579 , 0.00579505, 0.15542857, 0.00611739,
       0.70184095, 0.05332881, 0.65716745, 0.15458279, 0.34942711,
       0.00573833, 0.00297992, 0.00371618, 0.01549289, 0.00311902,
       0.61811096, 0.48895726, 0.00253995, 0.3372086 , 0.29996204,
       0.28344504, 0.01238464, 0.0063025 , 0.65716745, 0.00195777,
       0.15933834, 0.25683345, 0.03648573, 0.0111544 , 0.21936537])

In [67]:
world.features_arr.shape

(206, 30)

In [68]:
world.ShowFeatures()

### Tracks

In [69]:
world.ShowGridWorld_Count()

In [70]:
world.ShowGridWorld_Activated()

In [71]:
df_cluster = pd.read_csv(r'wifi_track_data\dacang\cluster_data\dacang_track_data_resident.csv')
world.experts.ReadCluster(df_cluster)
world.experts.df_trajs_all

,m,trajs,cluster
0,"7.5-48,74,38,133,29,213+1","[[420, 1], [460, 0], [460, 3], [459, 0], [459,...",-1
1,"7.5-48,74,38,133,29,213+2","[[877, 3], [876, 0], [876, 3], [875, 3], [874,...",-1
2,"7.5-48,74,38,155,214,98+1","[[420, 1], [460, 0], [460, 3], [459, 0], [459,...",-1
3,"7.5-48,74,38,155,214,98+2","[[877, 0], [877, 0], [877, 0], [877, 0], [877,...",-1
4,"7.6-48,74,38,155,214,98+13","[[788, 4], [789, 0], [789, 4], [790, 1], [830,...",0
...,...,...,...
7348,"8.7-124,161,119,201,50,40+22","[[743, 0], [743, 0], [743, 0], [743, 0], [743,...",-1
7349,"8.7-144,173,247,200,14,93+1","[[832, 3], [831, 3], [830, 0], [830, 3], [829,...",-1
7350,"8.7-144,173,247,200,14,93+2","[[374, 0], [374, 0], [374, 0], [374, 0], [374,...",-1
7351,"8.7-176,70,146,156,89,69+12","[[788, 4], [789, 0], [789, 1], [829, 4], [830,...",2


In [72]:
world.experts.ApplyCluster([-1])

In [73]:
world.experts.df_trajs

,m,trajs,cluster
0,"7.5-48,74,38,133,29,213+1","[[420, 1], [460, 0], [460, 3], [459, 0], [459,...",-1
1,"7.5-48,74,38,133,29,213+2","[[877, 3], [876, 0], [876, 3], [875, 3], [874,...",-1
2,"7.5-48,74,38,155,214,98+1","[[420, 1], [460, 0], [460, 3], [459, 0], [459,...",-1
3,"7.5-48,74,38,155,214,98+2","[[877, 0], [877, 0], [877, 0], [877, 0], [877,...",-1
4,"7.6-8,74,207,99,147,223+1","[[832, 3], [831, 3], [830, 0], [830, 3], [829,...",-1
...,...,...,...
3604,"8.7-124,161,119,201,50,40+20","[[743, 1], [783, 4], [784, 0], [784, 0], [784,...",-1
3605,"8.7-124,161,119,201,50,40+21","[[788, 3], [787, 2], [747, 0], [747, 1], [787,...",-1
3606,"8.7-124,161,119,201,50,40+22","[[743, 0], [743, 0], [743, 0], [743, 0], [743,...",-1
3607,"8.7-144,173,247,200,14,93+1","[[832, 3], [831, 3], [830, 0], [830, 3], [829,...",-1


### previous world

In [74]:
#env_folder_path = r'wifi_track_data/dacang/grid_data/env_imgs/40_30'
feature_folder_path = r"wifi_track_data\dacang\grid_data\features_grid\0918_40x30"
expert_traj_path = r"wifi_track_data\dacang\track_data\trajs_sliced_40x30.csv"

world_pre = GridWorld(features_folderPath=feature_folder_path,
                  expert_traj_filePath= expert_traj_path,
                  width=40,height=30)

Get experts trajs num:7353
trajs all avg length: 23


In [75]:
world_pre.features_arr.shape

(206, 30)

In [76]:
world_pre.ShowFeatures()